# Multimodal Reranker with Qwen3-VL and OpenVINO

The [Qwen3-VL-Reranker model series](https://huggingface.co/Qwen/Qwen3-VL-Reranker-8B) is built upon the powerful Qwen3-VL foundation model, designed to refine retrieval results by computing precise relevance scores for (query, document) pairs. Both query and document may contain arbitrary single or mixed modalities (text, images, screenshots, videos). In retrieval pipelines, the reranker is typically used in tandem with the embedding model: the embedding model performs efficient initial recall, while the reranker refines results in a subsequent re-ranking stage.

<img src="https://model-demo.oss-cn-hangzhou.aliyuncs.com/Qwen3-VL-Reranker.png" width="500"/>

In this tutorial we consider how to convert and optimize Qwen3-VL Reranker model using OpenVINO.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Select model](#Select-model)
- [Convert model using Optimum Intel](#Convert-model-using-Optimum-Intel)
- [Run OpenVINO model inference with Optimum-intel](#Run-OpenVINO-model-inference-with-Optimum-intel)

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/qwen3-vl-embedding/qwen3-vl-reranker.ipynb" />

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO and is using a custom branch of optimum-intel. It may be fully supported and validated in the future.

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
%pip uninstall -q -y optimum optimum-intel optimum-onnx
%pip install -q "git+https://github.com/openvino-dev-samples/optimum-intel.git@qwen3vl-reranker" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "transformers==5.8" "torch>=2.9" "torchvision" "qwen-vl-utils>=0.0.14" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -qU "openvino>=2025.4" "openvino_tokenizers>=2025.4"

In [2]:
import requests
from pathlib import Path

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("qwen3-vl-reranker.ipynb")

## Select model
[back to top ⬆️](#Table-of-contents:)

Qwen3-VL Reranker Model list:

| Model Type | Models | Size | Layers | Sequence Length | Instruction Aware |
|---|---|---|---|---|---|
| Multimodal Reranking | [Qwen3-VL-Reranker-2B](https://huggingface.co/Qwen/Qwen3-VL-Reranker-2B) | 2B | 28 | 32K | Yes |
| Multimodal Reranking | [Qwen3-VL-Reranker-8B](https://huggingface.co/Qwen/Qwen3-VL-Reranker-8B) | 8B | 36 | 32K | Yes |

In [3]:
import ipywidgets as widgets

model_ids = ["Qwen/Qwen3-VL-Reranker-2B", "Qwen/Qwen3-VL-Reranker-8B"]

model_selector = widgets.Dropdown(
    options=model_ids,
    default=model_ids[0],
    description="Reranker Model:",
)

model_selector

Dropdown(description='Reranker Model:', options=('Qwen/Qwen3-VL-Reranker-2B', 'Qwen/Qwen3-VL-Reranker-8B'), va…

## Convert model using Optimum Intel
[back to top ⬆️](#Table-of-contents:)

For convenience, we will use OpenVINO integration with HuggingFace Optimum. [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) is the interface between the Transformers and Diffusers libraries and the different tools and libraries provided by Intel to accelerate end-to-end pipelines on Intel architectures.

Among other use cases, Optimum Intel provides a simple interface to optimize your Transformers and Diffusers models, convert them to the OpenVINO Intermediate Representation (IR) format and run inference using OpenVINO Runtime. `optimum-cli` provides command line interface for model conversion and optimization.

General command format:

```bash
optimum-cli export openvino --model <model_id_or_path> --task <task> <output_dir>
```

where task is task to export the model for. Additionally, you can specify weights compression using `--weight-format` argument with one of following options: `fp32`, `fp16`, `int8` and `int4`.

In [4]:
to_compress = widgets.Checkbox(
    value=False,
    description="Weight compression",
    disabled=False,
)

visible_widgets = [to_compress]

options = widgets.VBox(visible_widgets)

options

The Qwen3-VL-Reranker model can be exported by `image-text-to-text` task with Optimum-intel.

In [5]:
from pathlib import Path

model_id = model_selector.value

model_base_dir = Path(model_id.split("/")[-1])
additional_args = {"task": "image-text-to-text"}

if to_compress.value:
    model_dir = model_base_dir / "INT8"
    additional_args.update({"weight-format": "int8"})
else:
    model_dir = model_base_dir / "FP16"
    additional_args.update({"weight-format": "fp16"})

In [6]:
from cmd_helper import optimum_cli

if not model_dir.exists():
    optimum_cli(model_id, model_dir, additional_args=additional_args)

**Export command:**

`optimum-cli export openvino --model Qwen/Qwen3-VL-Reranker-2B Qwen3-VL-Reranker-2B\FP16 --task image-text-to-text --weight-format fp16`

## Run OpenVINO model inference with Optimum-intel
[back to top ⬆️](#Table-of-contents:)

Select device from dropdown list for running inference using OpenVINO.

In [7]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU"])
device

Dropdown(description='Device:', options=('CPU', 'GPU', 'AUTO'), value='CPU')

The Qwen3-VL-Reranker model can be loaded by class `OVModelForVisualCausalLM` with Optimum-intel. Reranking works by computing yes/no token probabilities from the model's logits.

In [8]:
from optimum.intel import OVModelForVisualCausalLM

model = OVModelForVisualCausalLM.from_pretrained(model_dir, device=device.value, export=False)

In [9]:
import torch
from transformers import AutoProcessor
from qwen_vl_utils import process_vision_info


def compute_reranker_score(model, processor, query, document, instruction="Retrieve images or text relevant to the user's query."):
    """Compute relevance score for a single (query, document) pair with multimodal support."""
    tokenizer = processor.tokenizer if hasattr(processor, "tokenizer") else processor
    token_false_id = tokenizer.convert_tokens_to_ids("no")
    token_true_id = tokenizer.convert_tokens_to_ids("yes")

    query_text = query if isinstance(query, str) else query.get("text", "")
    content = []
    doc_text = ""
    if isinstance(document, dict):
        if "image" in document:
            content.append({"type": "image", "image": document["image"]})
        doc_text = document.get("text", "")
    elif isinstance(document, str):
        doc_text = document

    user_content = content + [{"type": "text", "text": f"<Query>: {query_text}\n<Document>: {doc_text}"}]
    conversation = [
        {"role": "system", "content": [{"type": "text", "text": instruction}]},
        {"role": "user", "content": user_content},
    ]
    prompt = processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
    image_inputs = process_vision_info(conversation)[0]
    if image_inputs:
        inputs = processor(text=[prompt], images=image_inputs, return_tensors="pt")
    else:
        inputs = processor(text=[prompt], return_tensors="pt")

    outputs = model.forward(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        pixel_values=inputs.get("pixel_values"),
        image_grid_thw=inputs.get("image_grid_thw"),
        mm_token_type_ids=inputs.get("mm_token_type_ids"),
    )

    logits = outputs.logits[:, -1, :]
    true_logits = logits[:, token_true_id]
    false_logits = logits[:, token_false_id]
    scores = torch.stack([false_logits, true_logits], dim=1)
    scores = torch.nn.functional.softmax(scores, dim=1)
    return scores[:, 1].item()


processor = AutoProcessor.from_pretrained(model_id)

# Example from https://huggingface.co/Qwen/Qwen3-VL-Reranker-2B#using-transformers
query = "A woman playing with her dog on a beach at sunset."
documents = [
    {
        "text": "A woman shares a joyful moment with her golden retriever on a sun-drenched beach at sunset, as the dog offers its paw in a heartwarming display of companionship and trust."
    },
    {"image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"},
    {
        "text": "A woman shares a joyful moment with her golden retriever on a sun-drenched beach at sunset, as the dog offers its paw in a heartwarming display of companionship and trust.",
        "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg",
    },
]
instruction = "Retrieve images or text relevant to the user's query."

# Each (query, document) pair is an independent relevance judgement, so we process them
# one at a time rather than batching (avoids left/right padding ambiguity for the last-token logits).
scores = []
for doc in documents:
    score = compute_reranker_score(model, processor, query, doc, instruction)
    scores.append(score)

print(f"Query: {query}\n")
print(f"Scores: {[round(s, 4) for s in scores]}")

# Reference scores for this example:
#   PyTorch BF16 (HF model card): [0.8613, 0.6757, 0.8125]
#   PyTorch FP32 (local run)    : [0.8522, 0.5560, 0.8477]
# OpenVINO FP16 matches PyTorch FP32 within FP16 tolerance. The larger gap versus the
# BF16 numbers reported on the model card comes from PyTorch dtype differences rather
# than from OpenVINO inference.

print("\nRanked results:")
for doc, score in sorted(zip(documents, scores), key=lambda x: x[1], reverse=True):
    print(f"  Score: {score:.4f} | {doc}")

Query: A woman playing with her dog on a beach at sunset.

Scores: [0.8522, 0.563, 0.8458]

Ranked results:
  Score: 0.8522 | {'text': 'A woman shares a joyful moment with her golden retriever on a sun-drenched beach at sunset, as the dog offers its paw in a heartwarming display of companionship and trust.'}
  Score: 0.8458 | {'text': 'A woman shares a joyful moment with her golden retriever on a sun-drenched beach at sunset, as the dog offers its paw in a heartwarming display of companionship and trust.', 'image': 'https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg'}
  Score: 0.5630 | {'image': 'https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg'}


### Compare with original PyTorch model (optional)
[back to top ⬆️](#Table-of-contents:)

Tick the checkbox below to also load the original PyTorch FP32 model and print the two score lists side-by-side. OpenVINO FP16 should agree with PyTorch FP32 within FP16 tolerance (differences of a few hundredths on this example).

> **Note:** this cell is **disabled by default** because loading the full FP32 model requires ~8 GB of memory and adds a few minutes of runtime.


In [10]:
import ipywidgets as widgets

run_pt_compare_widget = widgets.Checkbox(
    value=False,
    description="Run PyTorch FP32 comparison",
    disabled=False,
)

run_pt_compare_widget

Checkbox(value=False, description='Run PyTorch FP32 comparison')

In [12]:
if not run_pt_compare_widget.value:
    print("Skipping PyTorch comparison. Tick the checkbox above and re-run this cell to enable.")
else:
    import inspect

    from transformers import Qwen3VLForConditionalGeneration

    def compute_reranker_score_pt(model, processor, query, document, instruction="Retrieve images or text relevant to the user's query."):
        """PyTorch variant: drops kwargs the installed transformers version does not accept."""
        tokenizer = processor.tokenizer if hasattr(processor, "tokenizer") else processor
        token_false_id = tokenizer.convert_tokens_to_ids("no")
        token_true_id = tokenizer.convert_tokens_to_ids("yes")

        query_text = query if isinstance(query, str) else query.get("text", "")
        content = []
        doc_text = ""
        if isinstance(document, dict):
            if "image" in document:
                content.append({"type": "image", "image": document["image"]})
            doc_text = document.get("text", "")
        elif isinstance(document, str):
            doc_text = document

        user_content = content + [{"type": "text", "text": f"<Query>: {query_text}\n<Document>: {doc_text}"}]
        conversation = [
            {"role": "system", "content": [{"type": "text", "text": instruction}]},
            {"role": "user", "content": user_content},
        ]
        prompt = processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
        image_inputs = process_vision_info(conversation)[0]
        if image_inputs:
            inputs = processor(text=[prompt], images=image_inputs, return_tensors="pt")
        else:
            inputs = processor(text=[prompt], return_tensors="pt")

        accepted = set(inspect.signature(model.forward).parameters)
        call_kwargs = {k: v for k, v in inputs.items() if k in accepted}

        with torch.no_grad():
            outputs = model(**call_kwargs)

        logits = outputs.logits[:, -1, :]
        scores_pt = torch.stack([logits[:, token_false_id], logits[:, token_true_id]], dim=1)
        scores_pt = torch.nn.functional.softmax(scores_pt, dim=1)
        return scores_pt[:, 1].item()

    pt_model = Qwen3VLForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float32).eval()

    pt_scores = []
    for doc in documents:
        pt_scores.append(compute_reranker_score_pt(pt_model, processor, query, doc, instruction))

    print(f"PyTorch FP32 scores : {[round(s, 4) for s in pt_scores]}")
    print(f"OpenVINO FP16 scores: {[round(s, 4) for s in scores]}")

    max_abs_diff = max(abs(a - b) for a, b in zip(pt_scores, scores))
    print(f"Max |PT - OV| across the 3 documents: {max_abs_diff:.4f}")

    del pt_model
    import gc

    gc.collect()

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

PyTorch FP32 scores : [0.8522, 0.563, 0.8458]
OpenVINO FP16 scores: [0.8522, 0.563, 0.8458]
Max |PT - OV| across the 3 documents: 0.0000
